# Stage 8 on Kaggle — PySpark benchmark on Linux

Windows fought this stage for three separate reasons, none of them about
Spark: the Microsoft Store `python` alias, the absence of `fork` for worker
processes, and Hadoop's `winutils.exe`. Linux has none of them.

**Nothing about running it here weakens the result.** The deliverable is a
throughput measurement and a correctness proof, not the operating system
underneath. "Benchmarked on Linux" is a normal sentence.

### Setup, once

1. **Add your data.** *File → Add Data → New Dataset → Upload*, and include:
   - `data/interim/sections.parquet`
   - `spark_features.py`
   - `text_features.py`
   - `config.py`
   
   Name it `edgar-sections`. If it lands under a different name, fix the
   `SRC` path in the next cell.

2. **Settings → Accelerator → None.** Spark here is CPU-only; a GPU just
   costs you quota.

3. Run all.

In [ ]:
!pip install -q pyspark==3.5.3 pysentiment2==0.1.1 2>&1 | tail -2
import os, shutil, glob, sys
from pathlib import Path

SRC = Path("/kaggle/input/edgar-sections")
WORK = Path("/kaggle/working")

if not SRC.exists():
    cands = list(Path("/kaggle/input").glob("*"))
    raise SystemExit(f"dataset not found. Available inputs: {[c.name for c in cands]}\n"
                     "Point SRC at the right one.")

for name in ("spark_features.py", "text_features.py", "config.py"):
    src = next(iter(SRC.rglob(name)), None)
    if src is None:
        raise SystemExit(f"{name} is missing from the dataset — re-upload it")
    shutil.copy(src, WORK / name)

# config.py resolves data paths relative to itself; point it at the dataset.
parquet = next(iter(SRC.rglob("sections.parquet")), None)
if parquet is None:
    raise SystemExit("sections.parquet is missing from the dataset")
(WORK / "data" / "interim").mkdir(parents=True, exist_ok=True)
shutil.copy(parquet, WORK / "data" / "interim" / "sections.parquet")
os.environ["EDGAR_DATA_ROOT"] = str(WORK / "data")

sys.path.insert(0, str(WORK))
print("ready:", [p.name for p in WORK.glob("*.py")])
print("cpus:", os.cpu_count())

## Correctness first

Spark and pandas must produce the same numbers. A faster pipeline that
disagrees with the one it replaces is not a port — it is a second unvalidated
pipeline, and the disagreement surfaces later as a modelling result nobody can
explain.

Expect every LM feature to read `match` and `risk_change` to show **max abs
diff 0.0000**.

In [ ]:
import importlib, spark_features
importlib.reload(spark_features)
from spark_features import check

check(sample=300, cores=4)

## The benchmark

`spark_vs_pandas` above 1 means Spark is **slower** at that size — you are
paying JVM startup, serialisation and shuffle costs for nothing. The number
worth quoting is where that ratio crosses 1.

`--replicate` duplicates the corpus to reach sizes the real data cannot, which
is how the crossover gets found on one machine. Duplicated text is fine for a
throughput measurement and useless for a modelling one, which is why it is
only used here.

In [ ]:
from spark_features import benchmark

out = benchmark([500, 2000, 7000], replicate=1, cores=4)
out

In [ ]:
# Push past the real corpus size to find the crossover.
big = benchmark([7000], replicate=4, cores=4)
big

## What to record

| what | why |
|---|---|
| `spark_vs_pandas` at 7,000 filings | the honest "Spark cost us N x" number |
| the size where that ratio crosses 1 | the crossover — the actual finding |
| `arrow_speedup` | pandas_udf vs plain udf, measured rather than asserted |
| correctness `max abs diff` | that the port agrees with what it replaced |

Download `spark_benchmark.csv` from the output panel on the right.

### The sentence this stage buys

> *"At 8,801 filings Spark cost roughly N times more than pandas. Here is the
> measurement, and here is the corpus size at which it flips."*

An engineer who can say that is more useful than one who reaches for Spark by
default — which is the whole point of having done it.

In [ ]:
import pandas as pd
res = pd.read_csv("/kaggle/working/data/processed/spark_benchmark.csv")
print(res.to_string(index=False))
print()
crossed = res[res.spark_vs_pandas <= 1]
if len(crossed):
    print(f"Spark overtakes pandas at ~{int(crossed.filings.min()):,} filings")
else:
    print("Spark is still slower at every size tested — raise --replicate")